# MEME FIMO Motif Scan

![MEME FIMO Motif Scan](https://proto-bio.github.io/proto-assets/images/tool/meme/hero.png)

`run_meme_fimo_scan` uses MEME Suite [FIMO](https://meme-suite.org/meme/doc/fimo.html) (Find Individual Motif Occurrences) to scan sequences for occurrences of known motifs (position weight matrices) supplied in a [MEME](https://meme-suite.org/meme/) `.meme` file. It is backed by [`pymemesuite`](https://pypi.org/project/pymemesuite/), so no host MEME installation is required.

FIMO is described in Grant, Bailey & Noble (2011), ["FIMO: scanning for occurrences of a given motif"](https://doi.org/10.1093/bioinformatics/btr064), *Bioinformatics* 27(7):1017-1018.

In [ ]:
from pathlib import Path

import proto_tools
from proto_tools.tools.gene_annotation.meme import (
    MEMEFimoScanInput,
    MEMEFimoScanConfig,
    run_meme_fimo_scan,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [ ]:
# Display input docs
display_api_reference("meme-fimo-scan", "input", "run_meme_fimo_scan")

# Bundled MEME motif: the S. pyogenes SF370 CRISPR1 direct repeat (derived from the 7
# repeat instances in the locus below).
_meme_dir = Path(proto_tools.__file__).resolve().parent / "tools" / "gene_annotation" / "meme" / "examples"
motif_path = _meme_dir / "example.meme"

# Scan the real CRISPR1 locus for occurrences of the repeat motif.
dna_seq = "".join(l for l in (_meme_dir / "spyogenes_crispr_locus.fasta").read_text().splitlines() if not l.startswith(">"))

inputs = MEMEFimoScanInput(
    sequences=dna_seq,
    motifs=str(motif_path),
)

In [ ]:
# Display config docs
display_api_reference("meme-fimo-scan", "config", "run_meme_fimo_scan")

# Default config: p-value threshold 1e-4, scanning both strands | see docs above for all fields
config = MEMEFimoScanConfig()

In [ ]:
# Display output docs
display_api_reference("meme-fimo-scan", "output", "run_meme_fimo_scan")

## Basic usage

Scan the DNA sequence against the example motif with the default configuration. The two consensus copies should each produce a match.

In [ ]:
result = run_meme_fimo_scan(inputs, config)

print(f"Found {result.num_matches} motif occurrence(s)")

for match in result.results[0].matches:
    print(
        f"  motif={match.motif_id}, start={match.start}, stop={match.stop}, "
        f"strand={match.strand}, pvalue={match.pvalue:.2e}"
    )

## Advanced usage

Set `both_strands=False` to scan only the forward strand (FIMO `--norc`), and loosen `threshold` to report weaker matches.

In [ ]:
advanced_config = MEMEFimoScanConfig(
    threshold=1e-3,
    both_strands=False,
)

advanced_result = run_meme_fimo_scan(inputs, advanced_config)

print(f"Found {advanced_result.num_matches} motif occurrence(s) on the forward strand")

for match in advanced_result.results[0].matches:
    print(
        f"  motif={match.motif_id}, start={match.start}, stop={match.stop}, "
        f"strand={match.strand}, pvalue={match.pvalue:.2e}, matched={match.matched_sequence}"
    )

## Export results

FIMO matches can be exported to CSV (one row per match) or JSON. CSV is the default format.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="meme_fimo_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")